# Matrix Function via Eigenvalues

Chapter 9: Matrix Decompositions and Beyond · *Mathematics for AI and Machine Learning* by Fuheng Wu

Compare f(A) from spectral mapping with direct series-style intuition.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-9/matrix-function-eigs/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.set_printoptions(precision=5)

# The demo's default sliders: eigenvalues and the angle of the second eigenvector
l1, l2, angle_deg = 1.1, -0.4, 55
t = math.radians(angle_deg)
P = torch.tensor([[1.0, math.cos(t)],
                  [0.0, math.sin(t)]], dtype=torch.float64)
Pinv = torch.linalg.inv(P)
A = P @ torch.diag(torch.tensor([l1, l2], dtype=torch.float64)) @ Pinv

# ── f(A) = P f(D) P^{-1}: apply f to the eigenvalues only ───────────────────
expA_spectral = P @ torch.diag(torch.tensor([math.exp(l1), math.exp(l2)], dtype=torch.float64)) @ Pinv
expA_torch = torch.linalg.matrix_exp(A)


def exp_series(A, terms=30):
    """exp(A) = I + A + A^2/2! + ... (fine for small ||A||)."""
    out = torch.eye(2, dtype=A.dtype)
    term = torch.eye(2, dtype=A.dtype)
    for k in range(1, terms):
        term = term @ A / k
        out = out + term
    return out


print("exp(A) spectral =\n", expA_spectral)
print("matches torch.linalg.matrix_exp:", torch.allclose(expA_spectral, expA_torch))
print("matches the power series       :", torch.allclose(exp_series(A), expA_torch))

# ── Consequences: eigenvalues map, det(exp A) = exp(trace A) ────────────────
print("\neigenvalues of exp(A):", sorted(torch.linalg.eigvals(expA_torch).real.tolist(), reverse=True),
      " exp(l1), exp(l2) =", [math.exp(l1), math.exp(l2)])
print("det(exp A) =", torch.linalg.det(expA_torch).item(), " exp(trace A) =", math.exp(A.trace().item()))

# Any function works the same way, e.g. sin and cos
def f_of_A(f):
    return P @ torch.diag(torch.tensor([f(l1), f(l2)], dtype=torch.float64)) @ Pinv


sinA, cosA = f_of_A(math.sin), f_of_A(math.cos)
print("sin(A)^2 + cos(A)^2 == I:", torch.allclose(sinA @ sinA + cosA @ cosA, torch.eye(2, dtype=A.dtype)))

exp(A) spectral =
 tensor([[ 3.00417, -1.63418],
        [ 0.00000,  0.67032]], dtype=torch.float64)
matches torch.linalg.matrix_exp: True
matches the power series       : True

eigenvalues of exp(A): [3.004166023946434, 0.6703200460356391]  exp(l1), exp(l2) = [3.0041660239464334, 0.6703200460356393]
det(exp A) = 2.0137527074704766  exp(trace A) = 2.0137527074704766
sin(A)^2 + cos(A)^2 == I: True
